# Phase 8: model vs the line (Gate 2)

There were no free historical prop lines (OddsPapi history starts Jan 2026 and returned 404), so Gate 2 is measured **forward**, as the roadmap allows. Every DraftKings points line the pipeline sees (`nba/scripts/run_pipeline.py`) gets a model prediction, and once the game is final its actual points are stored in `nba/data/live/predictions.parquet`.

**Gate 2 passes if**, on the same player-games, model MAE <= line MAE **and** model P(over) log loss < the line's no-vig log loss. The line is treated as the bookmaker's prediction. We use the latest pre-game line (the closest thing to a closing line), because that is the hardest bar.

In [ ]:
import numpy as np
import pandas as pd
from nba_props.config import LIVE_DIR
from nba_props.evaluate import gate2
from nba_props.odds import american_to_decimal, no_vig_over

pd.set_option("display.max_columns", 50)
path = LIVE_DIR / "predictions.parquet"
preds = pd.read_parquet(path) if path.exists() else pd.DataFrame()
print(len(preds), "player-games seen")
done = preds[preds["played"] == True].copy() if "played" in preds else pd.DataFrame()  # noqa: E712
print(len(done), "finished with the player on the floor")
if len(preds):
    print(preds["start_time"].min(), "to", preds["start_time"].max())

## Step 1: Gate 2 on the latest pre-game line

In [ ]:
g_close = gate2(done) if len(done) else {"n": 0}
pd.Series(g_close)

## Step 2: the same check on the first line we saw
The first look is softer than the close. If the model only beats the first line, the edge is in betting early.

In [ ]:
if len(done):
    first = done.assign(line=done["first_line"], over_odds=done["first_over_odds"], under_odds=done["first_under_odds"],
                        predicted_points=done["first_predicted_points"], p_over=done["first_p_over"])
    g_first = gate2(first)
else:
    g_first = {"n": 0}
pd.DataFrame({"latest line": g_close, "first line": g_first})

## Step 3: calibration side by side
When the model says 60% over, does it go over 60% of the time? Same question for the bookmaker's no-vig price. Needs a few hundred rows before the bins mean anything.

In [ ]:
MIN_ROWS = 200
if len(done) >= MIN_ROWS:
    d = done[done["actual_points"] != done["line"]].copy()
    d["over"] = (d["actual_points"] > d["line"]).astype(int)
    d["p_book"] = no_vig_over(american_to_decimal(d["over_odds"]), american_to_decimal(d["under_odds"]))
    bins = np.linspace(0, 1, 11)
    rel = pd.concat({
        "model": d.groupby(pd.cut(d["p_over"], bins), observed=True).agg(pred=("p_over", "mean"), actual=("over", "mean"), rows=("over", "size")),
        "book": d.groupby(pd.cut(d["p_book"], bins), observed=True).agg(pred=("p_book", "mean"), actual=("over", "mean"), rows=("over", "size")),
    }, axis=1)
    display(rel.round(3))
else:
    print(f"only {len(done)} finished rows; calibration needs {MIN_ROWS}+")

## Step 4: where does the model miss most?
Lesson from Model 1: the biggest disagreements with the market are often the model's own errors. Compare errors by how far the model was from the line, and for opening games (last season's average filled in).

In [ ]:
if len(done):
    d = done.copy()
    d["gap"] = d["predicted_points"] - d["line"]
    d["model_err"] = (d["actual_points"] - d["predicted_points"]).abs()
    d["line_err"] = (d["actual_points"] - d["line"]).abs()
    d["gap_bucket"] = pd.cut(d["gap"].abs(), [0, 1, 2, 4, 100], include_lowest=True)
    display(d.groupby("gap_bucket", observed=True)[["model_err", "line_err"]].agg(["mean", "size"]).round(2))
    if "season_avg_filled" in d:
        display(d.groupby("season_avg_filled")[["model_err", "line_err"]].mean().round(2))
else:
    print("no finished props yet")

## Step 5: Gate 2 verdict
Write the numbers into `nba/docs/experiments.md`. Don't call it before ~500 finished player-games (about 2 weeks of the season).

In [ ]:
n = g_close.get("n", 0)
if n < 500:
    print(f"{n} player-games so far: too early for a verdict (aim for 500+)")
else:
    print("GATE 2", "PASSES" if g_close["passes"] else "FAILS", f"on {n} player-games")